In [2]:
import os
import pandas as pd
import json

# ===============================
# PATHS
# ===============================
data_mesh_path = "../Data_Mesh_Domains"  # Path to your domains
contracts_path = "../Data_Mesh_Domains/contracts"  # Path to contracts
validation_path = "../Validation_reports"
os.makedirs(validation_path, exist_ok=True)

# ===============================
# HELPER FUNCTIONS
# ===============================
def load_contract(domain_name):
    """Load YAML/JSON contract for the domain"""
    contract_file_json = os.path.join(contracts_path, f"{domain_name}_contract.json")
    if os.path.exists(contract_file_json):
        with open(contract_file_json) as f:
            contract = json.load(f)
        return contract
    else:
        return None

def validate_domain(domain_name, df, contract=None):
    """Run validation for a single domain"""
    results = {"domain": domain_name, "row_count": len(df), "issues": []}
    
    # 1. Schema compliance
    if contract:
        expected_columns = contract.get("columns", {}).keys()
        missing_cols = [c for c in expected_columns if c not in df.columns]
        extra_cols = [c for c in df.columns if c not in expected_columns]
        if missing_cols:
            results["issues"].append(f"Missing columns: {missing_cols}")
        if extra_cols:
            results["issues"].append(f"Extra columns: {extra_cols}")

    # 2. Nulls in non-nullable columns
    if contract:
        non_nullable = [col for col, props in contract.get("columns", {}).items() if not props.get("nullable", True)]
        null_issues = {col: df[col].isna().sum() for col in non_nullable if col in df.columns and df[col].isna().sum() > 0}
        if null_issues:
            results["issues"].append(f"Nulls in non-nullable columns: {null_issues}")
    
    # 3. Primary key uniqueness
    pk = contract.get("primary_key") if contract else df.columns[0]
    if pk in df.columns:
        duplicates = df[pk].duplicated().sum()
        if duplicates > 0:
            results["issues"].append(f"Primary key '{pk}' duplicates: {duplicates}")

    # 4. Basic value range checks (custom per domain)
    if "price_LKR" in df.columns:
        negative_prices = (df["price_LKR"] < 0).sum()
        if negative_prices > 0:
            results["issues"].append(f"Negative prices: {negative_prices}")

    if "stock_count" in df.columns:
        negative_stock = (df["stock_count"] < 0).sum()
        if negative_stock > 0:
            results["issues"].append(f"Negative stock_count: {negative_stock}")
    
    # Optional: add more domain-specific checks here
    
    return results

# ===============================
# LOAD DOMAINS
# ===============================
domain_data = {}
for domain_folder in os.listdir(data_mesh_path):
    domain_path = os.path.join(data_mesh_path, domain_folder)
    if os.path.isdir(domain_path):
        csv_files = [f for f in os.listdir(domain_path) if f.endswith(".csv")]
        if csv_files:
            df = pd.read_csv(os.path.join(domain_path, csv_files[0]))
            domain_data[domain_folder] = df
            print(f"Loaded {domain_folder}: {df.shape[0]} rows x {df.shape[1]} cols")

# ===============================
# RUN VALIDATION
# ===============================
validation_results = []

for domain_name, df in domain_data.items():
    contract = load_contract(domain_name)
    result = validate_domain(domain_name, df, contract)
    validation_results.append(result)

# ===============================
# SAVE VALIDATION REPORT
# ===============================
report_csv = os.path.join(validation_path, "data_mesh_validation_report.csv")
report_json = os.path.join(validation_path, "data_mesh_validation_report.json")


# Convert results to DataFrame for CSV
report_df = pd.DataFrame(validation_results)
report_df.to_csv(report_csv, index=False)

# Save JSON version
with open(report_json, "w") as f:
    json.dump(validation_results, f, indent=4)

print(f"✅ Validation reports saved:\nCSV: {report_csv}\nJSON: {report_json}")


Loaded users_domain: 508 rows x 7 cols
Loaded sales_domain: 8008 rows x 19 cols
Loaded user_preferences_domain: 471 rows x 11 cols
Loaded engagement_domain: 222 rows x 8 cols
Loaded customer_domain: 2856 rows x 6 cols
Loaded shop_domain: 24 rows x 9 cols
Loaded metadata: 7 rows x 6 cols
Loaded product_domain: 2483 rows x 13 cols
✅ Validation reports saved:
CSV: ../Validation_reports/data_mesh_validation_report.csv
JSON: ../Validation_reports/data_mesh_validation_report.json
